In [1]:
from pathlib import Path

BASE_DIR = (
    Path.home()
    / "Documentos"
    / "PUC"
    / "Projeto_Deep_Learning"
    / "data"
)

SOURCE_DIR = BASE_DIR / "BigEarthNet-S2"

print("Pasta existe?", SOURCE_DIR.exists())
print("Caminho:", SOURCE_DIR)

# Procura qualquer TIFF dentro da base
arquivos_tif = list(SOURCE_DIR.rglob("*.tif"))

print("Quantidade de TIFFs:", len(arquivos_tif))

# Mostra alguns exemplos
for arquivo in arquivos_tif[:10]:
    print(arquivo)

Pasta existe? True
Caminho: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/BigEarthNet-S2
Quantidade de TIFFs: 360000
/home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/BigEarthNet-S2/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL_42_90/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL_42_90_B02.tif
/home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/BigEarthNet-S2/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL_42_90/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL_42_90_B01.tif
/home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/BigEarthNet-S2/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL_42_90/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL_42_90_B09.tif
/home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/BigEarthNet-S2/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL/S2B_MSIL2A_20170709T094029_N9999_R036_T35VNL_42_90/S2B_MSIL2A_20170709T094029_N9999_

In [2]:
from pathlib import Path
import pandas as pd
import shutil


# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

BASE_DIR = (
    Path.home()
    / "Documentos"
    / "PUC"
    / "Projeto_Deep_Learning"
    / "data"
)

CSV_PATH = BASE_DIR / "selected_30k_spatial.csv"

SOURCE_DIR = BASE_DIR / "BigEarthNet-S2"

OUTPUT_DIR = BASE_DIR / "BigEarthNet-S2-10bands"


# ============================================================
# 2. BANDAS
# ============================================================

BANDAS = [
    "B02",
    "B03",
    "B04",
    "B05",
    "B06",
    "B07",
    "B08",
    "B8A",
    "B11",
    "B12",
]

SPLITS_VALIDOS = ["train", "validation", "test"]


# ============================================================
# 3. LEITURA DO CSV
# ============================================================

df = pd.read_csv(CSV_PATH)

df["patch_id"] = df["patch_id"].astype(str).str.strip()
df["split"] = df["split"].astype(str).str.strip().str.lower()

df = df[df["split"].isin(SPLITS_VALIDOS)].copy()

df_patches = df.drop_duplicates(
    subset=["patch_id"]
).copy()


# ============================================================
# 4. INDEXAR TODOS OS TIFFS DA BASE
# ============================================================

print("Indexando arquivos TIFF...")

arquivos_tif = list(SOURCE_DIR.rglob("*.tif"))

print(f"Total de TIFFs encontrados: {len(arquivos_tif)}")


# Criar índice pelo nome do arquivo
indice_tif = {
    arquivo.name: arquivo
    for arquivo in arquivos_tif
}


# ============================================================
# 5. CRIAR PASTAS DE SAÍDA
# ============================================================

for split in SPLITS_VALIDOS:
    (OUTPUT_DIR / split).mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 6. PROCESSAR PATCHES
# ============================================================

total_processados = 0
total_erros = 0

relatorio = []


for _, row in df_patches.iterrows():

    patch_id = row["patch_id"]
    split = row["split"]

    pasta_destino = OUTPUT_DIR / split / patch_id

    pasta_destino.mkdir(
        parents=True,
        exist_ok=True
    )

    bandas_copiadas = []
    bandas_faltantes = []

    for banda in BANDAS:

        nome_arquivo = f"{patch_id}_{banda}.tif"

        arquivo_origem = indice_tif.get(nome_arquivo)

        if arquivo_origem is None:

            bandas_faltantes.append(banda)

            continue

        arquivo_destino = pasta_destino / nome_arquivo

        shutil.copy2(
            arquivo_origem,
            arquivo_destino
        )

        bandas_copiadas.append(banda)

    if len(bandas_faltantes) == 0:

        status = "completo"
        total_processados += 1

    else:

        status = "bandas_faltantes"
        total_erros += 1

    relatorio.append({
        "patch_id": patch_id,
        "split": split,
        "status": status,
        "bandas_copiadas": len(bandas_copiadas),
        "bandas_faltantes": ",".join(bandas_faltantes),
    })


# ============================================================
# 7. RELATÓRIO
# ============================================================

df_relatorio = pd.DataFrame(relatorio)

RELATORIO_PATH = OUTPUT_DIR / "relatorio_bandas.csv"

df_relatorio.to_csv(
    RELATORIO_PATH,
    index=False
)


print("\n" + "=" * 60)
print("PROCESSAMENTO FINALIZADO")
print("=" * 60)

print("Patches completos:", total_processados)
print("Patches com erros:", total_erros)

print("\nStatus:")
print(df_relatorio["status"].value_counts())

print("\nRelatório:", RELATORIO_PATH)

Indexando arquivos TIFF...
Total de TIFFs encontrados: 360000

PROCESSAMENTO FINALIZADO
Patches completos: 30000
Patches com erros: 0

Status:
status
completo    30000
Name: count, dtype: int64

Relatório: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/BigEarthNet-S2-10bands/relatorio_bandas.csv
